# ParagAI: Train on Free Google Colab GPU (NVIDIA T4)

This notebook trains your custom **ParagAI** model from scratch on a free Google Colab GPU (NVIDIA T4).

### Step 0: Ensure GPU is Enabled
Go to **Runtime** > **Change runtime type** > Select **T4 GPU** > Click **Save**.

In [ ]:
# 1. Clone repository or pull latest commits
import os
if not os.path.exists('/content/ParagAI'):
    !git clone https://github.com/paragmm/ParagAI.git
%cd /content/ParagAI
!git switch -f feature/paragAI-pro
!git pull origin feature/paragAI-pro


In [ ]:
# 2. Install dependencies & register package in editable mode
!pip install -r requirements.txt
!pip install -e .

In [ ]:
# 3. Verify NVIDIA CUDA GPU
import torch
print("CUDA Available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU Device:", torch.cuda.get_device_name(0))
    print("Memory Allocated:", round(torch.cuda.get_device_properties(0).total_memory / (1024**3), 2), "GB")
else:
    print("WARNING: Running on CPU. Switch runtime type to T4 GPU for 10x faster training!")

In [ ]:
# 4. Prepare & merge multi-task SFT datasets (Chat, Translation, Automation)
!python scripts/prepare_data.py --no_tags


In [ ]:
# 5. Run Multi-Task Training on GPU (CUDA acceleration with BPE Tokenizer)
!python scripts/train.py --data data/processed/combined_corpus.txt --config configs/train_config.yaml --model_config configs/model_small.yaml --tokenizer_type bpe


In [ ]:
# 6. Test Interactive Chat & Multi-Task Generation in Python
import torch
from paragai.config import ModelConfig
from paragai.model.transformer import ParagAIModel
from paragai.tokenizer.bpe_tokenizer import BPETokenizer
from paragai.tokenizer.char_tokenizer import CharTokenizer
from paragai.inference.generate import Generator

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
checkpoint = torch.load('checkpoints/best_model.pt', map_location=device)
model_config = ModelConfig(**checkpoint['model_config'])
model = ParagAIModel(model_config)
state_dict = {k: v for k, v in checkpoint['model_state_dict'].items() if not k.endswith('.attn.mask')}
model.load_state_dict(state_dict, strict=False)

try:
    tokenizer = BPETokenizer.load('checkpoints/tokenizer.json')
except Exception:
    tokenizer = CharTokenizer.load('checkpoints/tokenizer.json')

generator = Generator(model, tokenizer, device=device)

# Interactive chat loop with ChatML prompt formatting
print('ParagAI Chat Interface Ready! (Type quit to exit)')
print('-' * 50)
while True:
    user_input = input('You: ')
    if user_input.strip().lower() in ('quit', 'exit'):
        break
    if not user_input.strip():
        continue
    prompt = f'<|im_start|>user\n{user_input}<|im_end|>\n<|im_start|>assistant\n'
    response = generator.generate(prompt, max_new_tokens=100, temperature=0.7, include_prompt=False)
    clean_response = response.split('<|im_end|>')[0].strip()
    print(f'ParagAI: {clean_response}\n')


In [ ]:
# 7. Export to GGUF format for Ollama (Optional)
!python scripts/export_hf.py --checkpoint checkpoints/best_model.pt --output_dir checkpoints/export_hf
!python scripts/export_gguf.py --input_dir checkpoints/export_hf --tokenizer checkpoints/tokenizer.json --output checkpoints/paragai.gguf

In [ ]:
# 8. Download trained model checkpoint & GGUF to your PC
from google.colab import files
files.download("checkpoints/best_model.pt")
files.download("checkpoints/tokenizer.json")
# If exported to GGUF:
# files.download("checkpoints/paragai.gguf")